<a href="https://colab.research.google.com/github/OlhaZahrebelna/certflow-rag-assistant/blob/main/src/retrieval/02_bm25_hybrid_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q rank-bm25

In [2]:
!pip install -q sentence-transformers faiss-cpu

In [3]:
import json
import re
from pathlib import Path
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [4]:
from pathlib import Path

repo_path = Path("/content/certflow-rag-assistant")

if not repo_path.exists():
    !git clone https://github.com/OlhaZahrebelna/certflow-rag-assistant.git
else:
    print("Repository already exists.")

Repository already exists.


In [5]:
%cd /content/certflow-rag-assistant

/content/certflow-rag-assistant


In [6]:
chunks_path = Path("data/raw/processed/chunks.json")

with open(chunks_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Loaded chunks: {len(chunks)}")

Loaded chunks: 81


In [7]:
def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"\b\w+\b", text)
    return tokens

In [8]:
tokenized_corpus = [
    tokenize(chunk["content"])
    for chunk in chunks
]

In [9]:
bm25 = BM25Okapi(tokenized_corpus)

In [10]:
def bm25_search(query, k=5):
    tokenized_query = tokenize(query)

    scores = bm25.get_scores(tokenized_query)

    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        chunk = chunks[idx]

        results.append({
            "rank": rank,
            "score": float(scores[idx]),
            "chunk_id": chunk["chunk_id"],
            "document": chunk["metadata"]["title"],
            "section": chunk["metadata"]["section"],
            "content": chunk["content"],
        })

    return results

In [11]:
results = bm25_search(
    "When should a certification case be escalated?",
    k=5
)

for result in results:
    print(
        result["rank"],
        round(result["score"], 4),
        result["document"],
        "→",
        result["section"]
    )

1 5.7593 Account Certification Frequently Asked Questions → Field decisions
2 5.2127 Source Hierarchy and Evidence Standard → 8. Unavailable sources
3 5.0578 Quality Review, Exceptions, and Escalations → 4. QA outcomes
4 4.5419 Roles and Responsibilities → 8. Segregation of duties
5 4.1538 Account Data Certification Overview → 5. Business principles


In [12]:
def evaluate_bm25_hit_at_k(evaluation_dataset, k=5):
    hits = 0
    details = []

    for item in evaluation_dataset:
        results = bm25_search(item["query"], k=k)

        is_hit = False
        hit_rank = None

        for result in results:
            for expected in item["expected_sections"]:
                if (
                    result["document"] == expected["document"]
                    and result["section"] == expected["section"]
                ):
                    is_hit = True
                    hit_rank = result["rank"]
                    break

            if is_hit:
                break

        if is_hit:
            hits += 1

        details.append({
            "query": item["query"],
            "hit": is_hit,
            "rank": hit_rank
        })

    return hits / len(evaluation_dataset), details

In [13]:
def evaluate_bm25_mrr(evaluation_dataset, k=5):
    reciprocal_ranks = []

    for item in evaluation_dataset:
        results = bm25_search(item["query"], k=k)

        rr = 0

        for result in results:
            for expected in item["expected_sections"]:
                if (
                    result["document"] == expected["document"]
                    and result["section"] == expected["section"]
                ):
                    rr = 1 / result["rank"]
                    break

            if rr > 0:
                break

        reciprocal_ranks.append(rr)

    return sum(reciprocal_ranks) / len(reciprocal_ranks)

In [14]:
evaluation_dataset = [
    {
        "query": "What evidence is required before an account can be certified?",
        "expected_sections": [
            {
                "document": "Source Hierarchy and Evidence Standard",
                "section": "6. Evidence recording"
            },
            {
                "document": "End-to-End Account Certification Workflow",
                "section": "4. Gather evidence"
            }
        ]
    },
    {
        "query": "How should potential duplicate accounts be handled?",
        "expected_sections": [
            {
                "document": "Address Certification and Duplicate Prevention",
                "section": "6. Duplicate screening"
            },
            {
                "document": "Address Certification and Duplicate Prevention",
                "section": "7. Duplicate outcomes"
            }
        ]
    },
    {
        "query": "When should a certification case be escalated?",
        "expected_sections": [
            {
                "document": "Quality Review, Exceptions, and Escalations",
                "section": "6. Escalation levels"
            }
        ]
    },
    {
        "query": "Who is responsible for performing the quality assurance review?",
        "expected_sections": [
            {
                "document": "Roles and Responsibilities",
                "section": "4. Quality Assurance Reviewer"
            }
        ]
    },
    {
        "query": "What validation rules apply to account fields?",
        "expected_sections": [
            {
                "document": "Account Fields and Validation Rules",
                "section": "2. Core field catalog"
            }
        ]
    },
    {
        "query": "What should an analyst do when two sources contain conflicting information?",
        "expected_sections": [
            {
                "document": "Source Hierarchy and Evidence Standard",
                "section": "5. Conflicting sources"
            }
        ]
    },
    {
        "query": "Which sources should be preferred when verifying account data?",
        "expected_sections": [
            {
                "document": "Source Hierarchy and Evidence Standard",
                "section": "2. Primary sources"
            },
            {
                "document": "Source Hierarchy and Evidence Standard",
                "section": "3. Secondary sources"
            }
        ]
    },
    {
        "query": "What should be recorded when a certification change is made?",
        "expected_sections": [
            {
                "document": "Request Types and Change Management",
                "section": "4. Required audit record"
            },
            {
                "document": "End-to-End Account Certification Workflow",
                "section": "6. Record the change"
            }
        ]
    },
    {
        "query": "What information is required when submitting a new certification request?",
        "expected_sections": [
            {
                "document": "Request Types and Change Management",
                "section": "2. Intake requirements"
            },
            {
                "document": "End-to-End Account Certification Workflow",
                "section": "1. Intake"
            }
        ]
    },
    {
        "query": "How do we verify that we are working with the correct company entity?",
        "expected_sections": [
            {
                "document": "End-to-End Account Certification Workflow",
                "section": "2. Identify the correct entity"
            }
        ]
    },
    {
        "query": "What checks are needed before setting an account to Verified?",
        "expected_sections": [
            {
                "document": "End-to-End Account Certification Workflow",
                "section": "9. Certification decision"
            }
        ]
    },
    {
        "query": "When is quality assurance mandatory?",
        "expected_sections": [
            {
                "document": "Quality Review, Exceptions, and Escalations",
                "section": "2. Mandatory QA triggers"
            }
        ]
    },
    {
        "query": "What should be checked during a QA review?",
        "expected_sections": [
            {
                "document": "Quality Review, Exceptions, and Escalations",
                "section": "3. QA checklist"
            }
        ]
    },
    {
        "query": "What happens when an account does not pass quality review?",
        "expected_sections": [
            {
                "document": "Quality Review, Exceptions, and Escalations",
                "section": "4. QA outcomes"
            }
        ]
    },
    {
        "query": "How should missing address information be handled?",
        "expected_sections": [
            {
                "document": "Address Certification and Duplicate Prevention",
                "section": "8. Missing address information"
            }
        ]
    },
    {
        "query": "How should an account address be normalized?",
        "expected_sections": [
            {
                "document": "Address Certification and Duplicate Prevention",
                "section": "5. Normalization"
            }
        ]
    },
    {
        "query": "What rules apply when the legal name of an account is updated?",
        "expected_sections": [
            {
                "document": "Account Fields and Validation Rules",
                "section": "3. Legal Name"
            }
        ]
    },
    {
        "query": "How should website and primary domain values be validated?",
        "expected_sections": [
            {
                "document": "Account Fields and Validation Rules",
                "section": "5. Website and Primary Domain"
            }
        ]
    },
    {
        "query": "What should happen when a relevant source is unavailable?",
        "expected_sections": [
            {
                "document": "Source Hierarchy and Evidence Standard",
                "section": "8. Unavailable sources"
            }
        ]
    },
    {
        "query": "When does a certified account need to be reviewed again?",
        "expected_sections": [
            {
                "document": "Account Data Certification Overview",
                "section": "6. Certification validity"
            }
        ]
    }
]

In [15]:
bm25_hit_rate, bm25_details = evaluate_bm25_hit_at_k(
    evaluation_dataset,
    k=5
)

bm25_mrr = evaluate_bm25_mrr(
    evaluation_dataset,
    k=5
)

print(f"BM25 Section Hit@5: {bm25_hit_rate:.2f}")
print(f"BM25 Section MRR@5: {bm25_mrr:.3f}")

BM25 Section Hit@5: 0.35
BM25 Section MRR@5: 0.233


### BM25 Baseline

BM25 was evaluated on the same 20-query retrieval dataset.

- Section Hit@5: 0.35
- Section MRR@5: 0.233

BM25 underperformed dense retrieval, indicating that semantic similarity is more important than exact lexical overlap for this knowledge base. BM25 will therefore be used as a complementary retriever in a hybrid retrieval pipeline rather than as a standalone retrieval method.

In [16]:
texts = [chunk["content"] for chunk in chunks]

dense_model = SentenceTransformer(
    "multi-qa-MiniLM-L6-cos-v1"
)

dense_embeddings = dense_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

dense_embeddings = np.asarray(
    dense_embeddings,
    dtype="float32"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

In [17]:
dimension = dense_embeddings.shape[1]

dense_index = faiss.IndexFlatIP(dimension)
dense_index.add(dense_embeddings)

print(f"Dense vectors indexed: {dense_index.ntotal}")

Dense vectors indexed: 81


In [18]:
def semantic_search(query, k=5):
    query_embedding = dense_model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    scores, indices = dense_index.search(
        query_embedding,
        k=k
    )

    results = []

    for rank, idx in enumerate(indices[0], start=1):
        chunk = chunks[idx]

        results.append({
            "rank": rank,
            "score": float(scores[0][rank - 1]),
            "chunk_id": chunk["chunk_id"],
            "document": chunk["metadata"]["title"],
            "section": chunk["metadata"]["section"],
            "content": chunk["content"],
        })

    return results

## RRF Hybrid Retrieval

In [19]:
def rrf_fusion(dense_results, bm25_results, k_rrf=60):
    fused_scores = {}
    result_lookup = {}

    # Dense / FAISS results
    for result in dense_results:
        chunk_id = result["chunk_id"]

        fused_scores[chunk_id] = (
            fused_scores.get(chunk_id, 0)
            + 1 / (k_rrf + result["rank"])
        )

        result_lookup[chunk_id] = result

    # BM25 results
    for result in bm25_results:
        chunk_id = result["chunk_id"]

        fused_scores[chunk_id] = (
            fused_scores.get(chunk_id, 0)
            + 1 / (k_rrf + result["rank"])
        )

        result_lookup[chunk_id] = result

    ranked_chunk_ids = sorted(
        fused_scores,
        key=fused_scores.get,
        reverse=True
    )

    fused_results = []

    for rank, chunk_id in enumerate(ranked_chunk_ids, start=1):
        result = result_lookup[chunk_id].copy()

        result["rank"] = rank
        result["rrf_score"] = fused_scores[chunk_id]

        fused_results.append(result)

    return fused_results

In [20]:
def hybrid_search(query, k=5, candidate_k=10):
    dense_results = semantic_search(
        query,
        k=candidate_k
    )

    lexical_results = bm25_search(
        query,
        k=candidate_k
    )

    fused_results = rrf_fusion(
        dense_results,
        lexical_results
    )

    return fused_results[:k]

In [21]:
results = hybrid_search(
    "When should a certification case be escalated?",
    k=5
)

for result in results:
    print(
        result["rank"],
        round(result["rrf_score"], 6),
        result["document"],
        "→",
        result["section"]
    )

1 0.031025 Quality Review, Exceptions, and Escalations → 4. QA outcomes
2 0.03101 Account Data Certification Overview → 5. Business principles
3 0.016393 Account Certification Frequently Asked Questions → General
4 0.016393 Account Certification Frequently Asked Questions → Field decisions
5 0.016129 Account Data Certification Overview → 4. Account certification outcomes


In [22]:
def evaluate_hybrid_hit_at_k(evaluation_dataset, k=5):
    hits = 0
    details = []

    for item in evaluation_dataset:
        results = hybrid_search(
            item["query"],
            k=k,
            candidate_k=10
        )

        is_hit = False
        hit_rank = None

        for result in results:
            for expected in item["expected_sections"]:
                if (
                    result["document"] == expected["document"]
                    and result["section"] == expected["section"]
                ):
                    is_hit = True
                    hit_rank = result["rank"]
                    break

            if is_hit:
                break

        if is_hit:
            hits += 1

        details.append({
            "query": item["query"],
            "hit": is_hit,
            "rank": hit_rank
        })

    return hits / len(evaluation_dataset), details

In [23]:
def evaluate_hybrid_mrr(evaluation_dataset, k=5):
    reciprocal_ranks = []

    for item in evaluation_dataset:
        results = hybrid_search(
            item["query"],
            k=k,
            candidate_k=10
        )

        rr = 0

        for result in results:
            for expected in item["expected_sections"]:
                if (
                    result["document"] == expected["document"]
                    and result["section"] == expected["section"]
                ):
                    rr = 1 / result["rank"]
                    break

            if rr > 0:
                break

        reciprocal_ranks.append(rr)

    return sum(reciprocal_ranks) / len(reciprocal_ranks)

In [24]:
hybrid_hit_rate, hybrid_details = evaluate_hybrid_hit_at_k(
    evaluation_dataset,
    k=5
)

hybrid_mrr = evaluate_hybrid_mrr(
    evaluation_dataset,
    k=5
)

print(f"Hybrid RRF Hit@5: {hybrid_hit_rate:.2f}")
print(f"Hybrid RRF MRR@5: {hybrid_mrr:.3f}")

Hybrid RRF Hit@5: 0.70
Hybrid RRF MRR@5: 0.422


In [25]:
print("Retrieval comparison")
print("-" * 60)

print(
    "Dense multi-qa + FAISS: "
    "Hit@5 = 0.85 | MRR@5 = 0.618"
)

print(
    "BM25: "
    "Hit@5 = 0.35 | MRR@5 = 0.233"
)

print(
    f"Hybrid RRF: "
    f"Hit@5 = {hybrid_hit_rate:.2f} | "
    f"MRR@5 = {hybrid_mrr:.3f}"
)

Retrieval comparison
------------------------------------------------------------
Dense multi-qa + FAISS: Hit@5 = 0.85 | MRR@5 = 0.618
BM25: Hit@5 = 0.35 | MRR@5 = 0.233
Hybrid RRF: Hit@5 = 0.70 | MRR@5 = 0.422


## Weighted RRF Experiment

In [26]:
def weighted_rrf_fusion(
    dense_results,
    bm25_results,
    dense_weight=0.8,
    bm25_weight=0.2,
    k_rrf=60
):
    fused_scores = {}
    result_lookup = {}

    # Dense retrieval
    for result in dense_results:
        chunk_id = result["chunk_id"]

        fused_scores[chunk_id] = (
            fused_scores.get(chunk_id, 0)
            + dense_weight / (k_rrf + result["rank"])
        )

        result_lookup[chunk_id] = result

    # BM25 retrieval
    for result in bm25_results:
        chunk_id = result["chunk_id"]

        fused_scores[chunk_id] = (
            fused_scores.get(chunk_id, 0)
            + bm25_weight / (k_rrf + result["rank"])
        )

        result_lookup[chunk_id] = result

    ranked_chunk_ids = sorted(
        fused_scores,
        key=fused_scores.get,
        reverse=True
    )

    fused_results = []

    for rank, chunk_id in enumerate(ranked_chunk_ids, start=1):
        result = result_lookup[chunk_id].copy()

        result["rank"] = rank
        result["weighted_rrf_score"] = fused_scores[chunk_id]

        fused_results.append(result)

    return fused_results

In [27]:
def weighted_hybrid_search(
    query,
    k=5,
    candidate_k=10,
    dense_weight=0.8,
    bm25_weight=0.2
):
    dense_results = semantic_search(
        query,
        k=candidate_k
    )

    lexical_results = bm25_search(
        query,
        k=candidate_k
    )

    fused_results = weighted_rrf_fusion(
        dense_results=dense_results,
        bm25_results=lexical_results,
        dense_weight=dense_weight,
        bm25_weight=bm25_weight
    )

    return fused_results[:k]

In [28]:
results = weighted_hybrid_search(
    "When should a certification case be escalated?",
    k=5
)

for result in results:
    print(
        result["rank"],
        round(result["weighted_rrf_score"], 6),
        result["document"],
        "→",
        result["section"]
    )

1 0.015577 Account Data Certification Overview → 5. Business principles
2 0.015296 Quality Review, Exceptions, and Escalations → 4. QA outcomes
3 0.013115 Account Certification Frequently Asked Questions → General
4 0.012903 Account Data Certification Overview → 4. Account certification outcomes
5 0.012698 Account Data Certification Overview → 3. Certification scope


In [29]:
def evaluate_weighted_hybrid_hit_at_k(
    evaluation_dataset,
    k=5,
    dense_weight=0.8,
    bm25_weight=0.2
):
    hits = 0
    details = []

    for item in evaluation_dataset:

        results = weighted_hybrid_search(
            item["query"],
            k=k,
            candidate_k=10,
            dense_weight=dense_weight,
            bm25_weight=bm25_weight
        )

        is_hit = False
        hit_rank = None

        for result in results:

            for expected in item["expected_sections"]:

                if (
                    result["document"] == expected["document"]
                    and result["section"] == expected["section"]
                ):
                    is_hit = True
                    hit_rank = result["rank"]
                    break

            if is_hit:
                break

        if is_hit:
            hits += 1

        details.append({
            "query": item["query"],
            "hit": is_hit,
            "rank": hit_rank
        })

    return hits / len(evaluation_dataset), details

In [30]:
def evaluate_weighted_hybrid_mrr(
    evaluation_dataset,
    k=5,
    dense_weight=0.8,
    bm25_weight=0.2
):
    reciprocal_ranks = []

    for item in evaluation_dataset:

        results = weighted_hybrid_search(
            item["query"],
            k=k,
            candidate_k=10,
            dense_weight=dense_weight,
            bm25_weight=bm25_weight
        )

        rr = 0

        for result in results:

            for expected in item["expected_sections"]:

                if (
                    result["document"] == expected["document"]
                    and result["section"] == expected["section"]
                ):
                    rr = 1 / result["rank"]
                    break

            if rr > 0:
                break

        reciprocal_ranks.append(rr)

    return sum(reciprocal_ranks) / len(reciprocal_ranks)

In [31]:
weighted_hit_rate, weighted_details = (
    evaluate_weighted_hybrid_hit_at_k(
        evaluation_dataset,
        k=5,
        dense_weight=0.8,
        bm25_weight=0.2
    )
)

weighted_mrr = evaluate_weighted_hybrid_mrr(
    evaluation_dataset,
    k=5,
    dense_weight=0.8,
    bm25_weight=0.2
)

print(f"Weighted RRF Hit@5: {weighted_hit_rate:.2f}")
print(f"Weighted RRF MRR@5: {weighted_mrr:.3f}")

Weighted RRF Hit@5: 0.75
Weighted RRF MRR@5: 0.488


In [32]:
print("Retrieval comparison")
print("-" * 70)

print(
    "Dense multi-qa + FAISS: "
    "Hit@5 = 0.85 | MRR@5 = 0.618"
)

print(
    "BM25: "
    "Hit@5 = 0.35 | MRR@5 = 0.233"
)

print(
    "Unweighted RRF: "
    "Hit@5 = 0.70 | MRR@5 = 0.422"
)

print(
    f"Weighted RRF (0.8 dense / 0.2 BM25): "
    f"Hit@5 = {weighted_hit_rate:.2f} | "
    f"MRR@5 = {weighted_mrr:.3f}"
)

Retrieval comparison
----------------------------------------------------------------------
Dense multi-qa + FAISS: Hit@5 = 0.85 | MRR@5 = 0.618
BM25: Hit@5 = 0.35 | MRR@5 = 0.233
Unweighted RRF: Hit@5 = 0.70 | MRR@5 = 0.422
Weighted RRF (0.8 dense / 0.2 BM25): Hit@5 = 0.75 | MRR@5 = 0.488


## Final Retrieval Strategy

Four retrieval strategies were evaluated on the same 20-query evaluation set.

| Retrieval Strategy                         |    Hit@5 |     MRR@5 |
| ------------------------------------------ | -------: | --------: |
| Dense: `multi-qa-MiniLM-L6-cos-v1` + FAISS | **0.85** | **0.618** |
| BM25                                       |     0.35 |     0.233 |
| Unweighted RRF                             |     0.70 |     0.422 |
| Weighted RRF (0.8 dense / 0.2 BM25)        |     0.75 |     0.488 |

Dense semantic retrieval achieved the best overall performance, with the highest retrieval coverage and ranking quality.

Weighted RRF improved over unweighted RRF, showing that the stronger dense retriever benefited from a higher contribution in the fusion step. However, adding BM25 still reduced performance compared with dense retrieval alone.

Based on these results, `multi-qa-MiniLM-L6-cos-v1` with FAISS was selected as the primary retrieval strategy for the next stage of the RAG pipeline.

The dense retrieval baseline (`Hit@5 = 0.85`, `MRR@5 = 0.618`) was established in `01_embeddings.ipynb` using the same 20-query evaluation set, ensuring a consistent comparison across retrieval approaches.
